# RAG Fundamentals + Observable Evaluation in SageMaker

**Track:** Applied Agent Engineering Foundations  
**Merged notebook:** RAG Fundamentals + Observable RAG Evaluation  
**Runtime target:** SageMaker Jupyter Notebook  
**Model access:** Amazon Bedrock  
**Data source:** S3  
**Supported input files:** `.docx`

This notebook combines the earlier **RAG Fundamentals** flow with the **Observable RAG Evaluation** flow into one classroom-ready lab.


## What learners will do

By the end of this notebook, learners will be able to:

- load `.docx`, `.txt`, and `.md` documents from S3
- compare chunking strategies and choose one for retrieval
- generate embeddings from a Bedrock embedding model available in the current region
- retrieve relevant chunks using in-memory similarity search
- optionally rerank retrieved chunks with an LLM
- generate grounded answers using Bedrock
- inspect retrieval traces, run metrics, and evaluation results in DataFrames
- improve the system during a mini-hack without saving any local artifacts



In [3]:
# Uncomment only if your environment needs these libraries.
%pip install -q boto3 pandas numpy python-docx


Note: you may need to restart the kernel to use updated packages.


## Step 1 — Configuration and imports

Update the S3 bucket and prefix before running the notebook.

**Instructor cue:**  
Explain that this notebook is intentionally simple enough for the classroom:
- Bedrock for model access
- S3 for source documents
- pandas DataFrames for traceability
- no local vector store


In [4]:
from __future__ import annotations

import io
import json
import os
import re
from typing import Any

import boto3
import numpy as np
import pandas as pd
from botocore.exceptions import ClientError
from docx import Document as DocxDocument
from IPython.display import display


In [16]:

AWS_REGION = os.environ.get("AWS_DEFAULT_REGION", "us-east-1")

# LLM model for answer generation. Available models for this class
"""
amazon.nova-lite-v1:0 
amazon.nova-micro-v1:0 
amazon.nova-pro-v1:0
"""
BEDROCK_LLM_MODEL_ID = "amazon.nova-lite-v1:0"

# Embedding model. Available embedding model for this class
"""
amazon.titan-embed-text-v1 
amazon.titan-embed-text-v2:0
amazon.titan-embed-image-v1
"""
BEDROCK_EMBEDDING_MODEL_ID = "amazon.titan-embed-text-v1"

# S3 bucket
S3_BUCKET = "s3-demo-bucket-adp"
S3_PREFIX = "rag-docs/"


ALLOWED_EXTENSIONS = (".docx")

CHUNK_SIZE = 700
CHUNK_OVERLAP = 120
TOP_K = 5
FINAL_K = 3

# Optional cap to make classroom runs faster. Set to None to embed every chunk.
MAX_CHUNKS_TO_EMBED = None

s3 = boto3.client("s3", region_name=AWS_REGION)
bedrock_runtime = boto3.client("bedrock-runtime", region_name=AWS_REGION)
bedrock = boto3.client("bedrock", region_name=AWS_REGION)

print("AWS Region:", AWS_REGION)
print("LLM model:", BEDROCK_LLM_MODEL_ID)
print("Requested embedding model:", BEDROCK_EMBEDDING_MODEL_ID or "AUTO-DISCOVER")
print("Allowed file types:", ALLOWED_EXTENSIONS)


AWS Region: us-east-1
LLM model: amazon.nova-lite-v1:0
Requested embedding model: amazon.titan-embed-text-v1
Allowed file types: .docx


## Step 3 — Load `.docx`, `.txt`, and `.md` documents from S3

This notebook reads directly from S3 and keeps the loaded corpus in memory.

**Discuss with learners:**
- production systems usually separate storage from compute
- S3 is the document source of truth here
- loaders are format-specific because `.docx` and `.txt` are not parsed the same way


In [18]:
import io  # Used to handle the .docx file in memory as a byte stream
import pandas as pd  # Used to store the loaded document details in a DataFrame
from docx import Document as DocxDocument  # Used to read Word (.docx) files

# This function lists all .docx files inside a given S3 bucket and prefix(folder path)
def list_document_keys(bucket: str, prefix: str) -> list[str]:
    # Ask S3 to list objects inside the given bucket and prefix
    response = s3.list_objects_v2(Bucket=bucket, Prefix=prefix)
    
    # Create an empty list to store matching file keys
    keys = []

    # Loop through all objects returned by S3
    for obj in response.get("Contents", []):
        # Extract the file path/key from each object
        key = obj["Key"]
        
        # Keep only files that end with .docx
        if key.lower().endswith(".docx"):
            keys.append(key)

    # Return the sorted list of document keys
    return sorted(keys)

# This function reads one .docx file from S3 and converts it into plain text
def read_docx_from_s3(bucket: str, key: str) -> str:
    # Download the file from S3
    response = s3.get_object(Bucket=bucket, Key=key)
    
    # Read the file content as raw bytes
    raw = response["Body"].read()

    # Open the .docx file from memory using BytesIO
    doc = DocxDocument(io.BytesIO(raw))
    
    # Create an empty list to collect paragraph text
    parts = []

    # Loop through each paragraph in the Word document
    for p in doc.paragraphs:
        # Remove extra spaces from paragraph text
        text = p.text.strip()
        
        # Add only non-empty paragraphs
        if text:
            parts.append(text)

    # Join all paragraphs into one text block separated by blank lines
    return "\n\n".join(parts)

# Get the list of .docx files from the given S3 bucket and prefix
document_keys = list_document_keys(S3_BUCKET, S3_PREFIX)

# Print how many documents were found
print("Document count:", len(document_keys))

# Print the first 10 file keys so we can inspect them
print(document_keys[:10])



Document count: 9
['rag-docs/HR-POL-001_Employee_Onboarding_Policy.docx', 'rag-docs/HR-POL-002_Equal_Opportunity_Policy.docx', 'rag-docs/HR-POL-003_Health_Benefits_Policy.docx', 'rag-docs/HR-POL-004_Training_and_Development_Policy.docx', 'rag-docs/HR-POL-005_Termination_and_Suspension_Policy.docx', 'rag-docs/HR-POL-006_Performance_Appraisal_Policy.docx', 'rag-docs/HR-POL-007_General_Work_Policy.docx', 'rag-docs/HR-POL-008_Leave_Policy.docx', 'rag-docs/HR-POL-009_Remote_Work_Policy.docx']


In [19]:
# Create an empty list to store all loaded documents
raw_documents = []

# Loop through each document key
for key in document_keys:
    # Read the text content from the current .docx file
    text = read_docx_from_s3(S3_BUCKET, key).strip()

    # Add the document only if text is not empty
    if text:
        raw_documents.append({
            "source": key,             # Full S3 file path
            "file_type": "docx",       # File type is fixed as docx
            "text": text,              # Extracted document text
            "char_len": len(text),     # Number of characters in the text
        })

# Convert the list of dictionaries into a pandas DataFrame
docs_df = pd.DataFrame(raw_documents)

# Display the DataFrame in notebook output
display(docs_df)

# Print the final number of loaded documents
print("Loaded documents:", len(docs_df))

,source,file_type,text,char_len
0,rag-docs/HR-POL-001_Employee_Onboarding_Policy...,docx,ABC Solutions Pvt. Ltd.\nDocument ID: HR-POL-0...,4869
1,rag-docs/HR-POL-002_Equal_Opportunity_Policy.docx,docx,ABC Solutions Pvt. Ltd.\n\nABC Solutions Pvt. ...,5318
2,rag-docs/HR-POL-003_Health_Benefits_Policy.docx,docx,ABC Solutions Pvt. Ltd.\n\nABC Solutions Pvt. ...,5159
3,rag-docs/HR-POL-004_Training_and_Development_P...,docx,ABC Solutions Pvt. Ltd.\n\nABC Solutions Pvt. ...,5329
4,rag-docs/HR-POL-005_Termination_and_Suspension...,docx,ABC Solutions Pvt. Ltd.\n\nABC Solutions Pvt. ...,5651
5,rag-docs/HR-POL-006_Performance_Appraisal_Poli...,docx,ABC Solutions Pvt. Ltd.\n\nDocument ID: HR-POL...,5418
6,rag-docs/HR-POL-007_General_Work_Policy.docx,docx,ABC Solutions Pvt. Ltd.\n\nABC Solutions Pvt. ...,4306
7,rag-docs/HR-POL-008_Leave_Policy.docx,docx,ABC Solutions Pvt. Ltd.\n\nABC Solutions Pvt. ...,3708
8,rag-docs/HR-POL-009_Remote_Work_Policy.docx,docx,ABC Solutions Pvt. Ltd.\n\nABC Solutions Pvt. ...,4137


Loaded documents: 9


## Step 4 — Compare chunking strategies

We compare three chunking approaches:

1. fixed-size chunking
2. fixed-size chunking with overlap
3. section-based chunking

**Instructor cue:**  
Use this section to help learners understand why chunking changes retrieval quality.


In [30]:
# This function splits a long text into smaller pieces called chunks
def basic_chunking(text: str, chunk_size: int = 500) -> list[str]:
    # Create chunks of fixed size by moving from the start of the text
    # to the end in steps of chunk_size
    return [text[i:i+chunk_size] for i in range(0, len(text), chunk_size)]

# Apply the chunking function to the text of the first document in docs_df
basic_chunking(docs_df.iloc[0]["text"],10000)

['ABC Solutions Pvt. Ltd.\nDocument ID: HR-POL-001\nVersion: 1.0\nEffective Date: January 1, 2026\nTitle: Employee Onboarding Policy\nClassification: Confidential – Internal Use Only\n\n1. Objective\n\nThis policy establishes a standardized, legally compliant, and auditable onboarding framework for all new hires of ABC Solutions Pvt. Ltd. The objective is to ensure timely statutory registration, background verification, information security clearance, role readiness, and cultural integration in accordance with the Industrial Employment (Standing Orders) Act, 1946, the Information Technology Act, 2000, the Employees’ Provident Funds and Miscellaneous Provisions Act, 1952, the Employees’ State Insurance Act, 1948, and applicable State Shops and Establishments Acts.\n\n2. Scope\n\nThis policy applies to all permanent, fixed-term, and contractual employees with an engagement period exceeding 90 calendar days, across all business units, grades (G1–G10), and locations of ABC Solutions Pvt. L

In [32]:
# This function splits text into chunks with overlap between consecutive chunks
def chunk_with_overlap(text: str, chunk_size: int = CHUNK_SIZE, overlap: int = CHUNK_OVERLAP) -> list[str]:
    
    # Ensure overlap is smaller than chunk size to avoid infinite loop or invalid logic
    if overlap >= chunk_size:
        raise ValueError("overlap must be smaller than chunk_size")
    
    # List to store all generated chunks
    chunks = []
    
    # Step size determines how much we move forward each time
    # Overlap means we move less than chunk_size
    step = chunk_size - overlap
    
    # Loop through the text using the step size
    for i in range(0, len(text), step):
        
        # Extract a chunk of size 'chunk_size' starting from index i
        chunk = text[i:i+chunk_size]
        
        # Add the chunk only if it is not empty
        if chunk:
            chunks.append(chunk)
    
    # Return the list of overlapping chunks
    return chunks

# Apply the function to the text of the first document
chunk_with_overlap(docs_df.iloc[0]["text"],100,5)


['ABC Solutions Pvt. Ltd.\nDocument ID: HR-POL-001\nVersion: 1.0\nEffective Date: January 1, 2026\nTitle: ',
 'tle: Employee Onboarding Policy\nClassification: Confidential – Internal Use Only\n\n1. Objective\n\nThis',
 '\nThis policy establishes a standardized, legally compliant, and auditable onboarding framework for a',
 'for all new hires of ABC Solutions Pvt. Ltd. The objective is to ensure timely statutory registratio',
 'ration, background verification, information security clearance, role readiness, and cultural integr',
 'ntegration in accordance with the Industrial Employment (Standing Orders) Act, 1946, the Information',
 'ation Technology Act, 2000, the Employees’ Provident Funds and Miscellaneous Provisions Act, 1952, t',
 '52, the Employees’ State Insurance Act, 1948, and applicable State Shops and Establishments Acts.\n\n2',
 's.\n\n2. Scope\n\nThis policy applies to all permanent, fixed-term, and contractual employees with an en',
 'an engagement period exceeding 90 cal

In [33]:
import re  # Used for splitting the text based on blank lines

# This function splits the text into sections using blank lines as separators
def section_based_chunking(text: str, min_section_chars: int = 120) -> list[str]:
    
    # Split the text wherever there are blank lines, then remove extra spaces
    sections = [sec.strip() for sec in re.split(r"\n\s*\n", text) if sec.strip()]
    
    # Keep only sections that are at least the minimum required length
    return [sec for sec in sections if len(sec) >= min_section_chars]

# Check whether the DataFrame has any loaded documents
if docs_df.empty:
    raise ValueError("No documents loaded from S3.")

# Apply section-based chunking to the text of the first document
section_based_chunking(docs_df.iloc[0]["text"],200)



['This policy establishes a standardized, legally compliant, and auditable onboarding framework for all new hires of ABC Solutions Pvt. Ltd. The objective is to ensure timely statutory registration, background verification, information security clearance, role readiness, and cultural integration in accordance with the Industrial Employment (Standing Orders) Act, 1946, the Information Technology Act, 2000, the Employees’ Provident Funds and Miscellaneous Provisions Act, 1952, the Employees’ State Insurance Act, 1948, and applicable State Shops and Establishments Acts.',
 'This policy applies to all permanent, fixed-term, and contractual employees with an engagement period exceeding 90 calendar days, across all business units, grades (G1–G10), and locations of ABC Solutions Pvt. Ltd. The policy is administered by the Human Resources Department under the authority of the Chief Human Resources Officer (CHRO) and is subject to operational oversight by the Chief Information Security Officer 

In [27]:
# Get the source path of the first document from the DataFrame
sample_key = docs_df.iloc[0]["source"]

# Get the text content of the first document from the DataFrame
sample_text = docs_df.iloc[0]["text"]

# Apply all three chunking methods to the sample text
chunking_comparison = {
    "basic": basic_chunking(sample_text),                  # Fixed-size chunks
    "overlap": chunk_with_overlap(sample_text),           # Fixed-size chunks with overlap
    "section": section_based_chunking(sample_text),       # Section-based chunks using blank lines
}

# Create an empty list to store summary information for each chunking method
comparison_rows = []

# Loop through each chunking method and its generated chunks
for method, chunks in chunking_comparison.items():
    comparison_rows.append({
        "method": method,                                 # Name of the chunking method
        "n_chunks": len(chunks),                          # Total number of chunks created
        "first_chunk_preview": chunks[0][:300] if chunks else ""  # First 300 characters of the first chunk
    })

# Convert the summary list into a DataFrame
chunking_comparison_df = pd.DataFrame(comparison_rows)

# Display the comparison table
display(chunking_comparison_df)

,method,n_chunks,first_chunk_preview
0,basic,10,ABC Solutions Pvt. Ltd.\nDocument ID: HR-POL-0...
1,overlap,9,ABC Solutions Pvt. Ltd.\nDocument ID: HR-POL-0...
2,section,14,ABC Solutions Pvt. Ltd.\nDocument ID: HR-POL-0...


## Step 5 — Build chunk records with metadata

For the rest of the notebook we will use **overlap chunking**.  
Each chunk keeps metadata so that retrieval is explainable.

**Discuss with learners:**
- retrieval is better when chunks carry source metadata
- metadata helps filtering, debugging, and governance


In [34]:
# Create an empty list to store all chunk-level rows
chunk_rows = []

# Loop through each document in docs_df
for row in docs_df.itertuples():
    
    # Apply overlapping chunking to the document text
    chunks = chunk_with_overlap(row.text, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP)
    
    # Loop through each chunk and also keep track of chunk number
    for i, chunk in enumerate(chunks, start=1):
        chunk_rows.append({
            "source": row.source,                              # Original document path/name
            "file_type": row.file_type,                        # File type of the document
            "chunk_id": f"{row.source}::chunk_{i:03d}",        # Unique chunk ID
            "chunk_index": i,                                  # Chunk number within the document
            "chunk_text": chunk,                               # Actual chunk text
            "char_len": len(chunk),                            # Number of characters in the chunk
        })

# Convert all chunk rows into a DataFrame
chunks_df = pd.DataFrame(chunk_rows)

# Display the first 10 chunk rows
display(chunks_df.head(10))

# Print total number of chunks created
print("Total chunk rows:", len(chunks_df))


,source,file_type,chunk_id,chunk_index,chunk_text,char_len
0,rag-docs/HR-POL-001_Employee_Onboarding_Policy...,docx,rag-docs/HR-POL-001_Employee_Onboarding_Policy...,1,ABC Solutions Pvt. Ltd.\nDocument ID: HR-POL-0...,700
1,rag-docs/HR-POL-001_Employee_Onboarding_Policy...,docx,rag-docs/HR-POL-001_Employee_Onboarding_Policy...,2,"nology Act, 2000, the Employees’ Provident Fun...",700
2,rag-docs/HR-POL-001_Employee_Onboarding_Policy...,docx,rag-docs/HR-POL-001_Employee_Onboarding_Policy...,3,Chief Information Security Officer (CISO) and...,700
3,rag-docs/HR-POL-001_Employee_Onboarding_Policy...,docx,rag-docs/HR-POL-001_Employee_Onboarding_Policy...,4,1 Offer and Pre-Joining Formalities\n4.1.1 Fin...,700
4,rag-docs/HR-POL-001_Employee_Onboarding_Policy...,docx,rag-docs/HR-POL-001_Employee_Onboarding_Policy...,5,V shall be initiated on DoJ and completed with...,700
5,rag-docs/HR-POL-001_Employee_Onboarding_Policy...,docx,rag-docs/HR-POL-001_Employee_Onboarding_Policy...,6,of a minimum of 16 hours of classroom sessions...,700
6,rag-docs/HR-POL-001_Employee_Onboarding_Policy...,docx,rag-docs/HR-POL-001_Employee_Onboarding_Policy...,7,", BGV, or security clearance within prescribed...",700
7,rag-docs/HR-POL-001_Employee_Onboarding_Policy...,docx,rag-docs/HR-POL-001_Employee_Onboarding_Policy...,8,"r HR-POL-009, Clause 3.1.\n7.4 Information Sec...",700
8,rag-docs/HR-POL-001_Employee_Onboarding_Policy...,docx,rag-docs/HR-POL-001_Employee_Onboarding_Policy...,9,"cuments, or violation of information security ...",229
9,rag-docs/HR-POL-002_Equal_Opportunity_Policy.docx,docx,rag-docs/HR-POL-002_Equal_Opportunity_Policy.d...,1,ABC Solutions Pvt. Ltd.\n\nABC Solutions Pvt. ...,700


Total chunk rows: 81


## Step 6 — Embedding




In [35]:
import json
import numpy as np

# Use the embedding model already defined earlier
EMBEDDING_MODEL_ID = BEDROCK_EMBEDDING_MODEL_ID

# This function calls Bedrock and returns embedding for given text
def get_text_embedding(text: str):
    response = bedrock_runtime.invoke_model(
        modelId=EMBEDDING_MODEL_ID,
        body=json.dumps({"inputText": text}),   # Titan expects inputText
        accept="application/json",
        contentType="application/json",
    )
    
    # Convert response to JSON and extract embedding vector
    return json.loads(response["body"].read())["embedding"]

# Generate embeddings for ALL chunks (no limit)
chunks_df["embedding"] = chunks_df["chunk_text"].apply(get_text_embedding)

# Store embedding size (dimension)
chunks_df["embedding_dim"] = chunks_df["embedding"].apply(len)

# Convert list of embeddings into a matrix (useful for similarity search later)
embedding_matrix = np.vstack(chunks_df["embedding"].values).astype("float32")

# Display sample rows (without full embedding vector)
display(chunks_df.drop(columns=["embedding"]).head(10))

# Print summary
print("Total chunks embedded:", len(chunks_df))
print("Embedding dimension:", chunks_df.iloc[0]["embedding_dim"])

,source,file_type,chunk_id,chunk_index,chunk_text,char_len,embedding_dim
0,rag-docs/HR-POL-001_Employee_Onboarding_Policy...,docx,rag-docs/HR-POL-001_Employee_Onboarding_Policy...,1,ABC Solutions Pvt. Ltd.\nDocument ID: HR-POL-0...,700,1536
1,rag-docs/HR-POL-001_Employee_Onboarding_Policy...,docx,rag-docs/HR-POL-001_Employee_Onboarding_Policy...,2,"nology Act, 2000, the Employees’ Provident Fun...",700,1536
2,rag-docs/HR-POL-001_Employee_Onboarding_Policy...,docx,rag-docs/HR-POL-001_Employee_Onboarding_Policy...,3,Chief Information Security Officer (CISO) and...,700,1536
3,rag-docs/HR-POL-001_Employee_Onboarding_Policy...,docx,rag-docs/HR-POL-001_Employee_Onboarding_Policy...,4,1 Offer and Pre-Joining Formalities\n4.1.1 Fin...,700,1536
4,rag-docs/HR-POL-001_Employee_Onboarding_Policy...,docx,rag-docs/HR-POL-001_Employee_Onboarding_Policy...,5,V shall be initiated on DoJ and completed with...,700,1536
5,rag-docs/HR-POL-001_Employee_Onboarding_Policy...,docx,rag-docs/HR-POL-001_Employee_Onboarding_Policy...,6,of a minimum of 16 hours of classroom sessions...,700,1536
6,rag-docs/HR-POL-001_Employee_Onboarding_Policy...,docx,rag-docs/HR-POL-001_Employee_Onboarding_Policy...,7,", BGV, or security clearance within prescribed...",700,1536
7,rag-docs/HR-POL-001_Employee_Onboarding_Policy...,docx,rag-docs/HR-POL-001_Employee_Onboarding_Policy...,8,"r HR-POL-009, Clause 3.1.\n7.4 Information Sec...",700,1536
8,rag-docs/HR-POL-001_Employee_Onboarding_Policy...,docx,rag-docs/HR-POL-001_Employee_Onboarding_Policy...,9,"cuments, or violation of information security ...",229,1536
9,rag-docs/HR-POL-002_Equal_Opportunity_Policy.docx,docx,rag-docs/HR-POL-002_Equal_Opportunity_Policy.d...,1,ABC Solutions Pvt. Ltd.\n\nABC Solutions Pvt. ...,700,1536


Total chunks embedded: 81
Embedding dimension: 1536


## Step 7 — Retrieval by similarity


In [39]:
import numpy as np
import pandas as pd

# Retrieve top_k relevant chunks for a query
def retrieve(query: str, top_k: int = TOP_K):

    # Convert query to embedding and normalize
    query_embedding = np.array(get_text_embedding(query), dtype="float32")
    query_embedding = query_embedding / np.linalg.norm(query_embedding)

    # Normalize all chunk embeddings and create matrix
    embedding_matrix = np.vstack([
        np.array(vec, dtype="float32") / np.linalg.norm(vec)
        for vec in chunks_df["embedding"]
    ])

    # Compute similarity scores
    scores = embedding_matrix @ query_embedding

    # Get top_k indices
    top_indices = np.argsort(-scores)[:top_k]

    # Collect results
    rows = []
    for rank, idx in enumerate(top_indices, start=1):
        row = chunks_df.iloc[idx]
        rows.append({
            "query": query,
            "rank": rank,
            "score": float(scores[idx]),
            "source": row["source"],
            "chunk_id": row["chunk_id"],
            "chunk_text": row["chunk_text"]
        })

    # Convert to DataFrame
    retrieval_df = pd.DataFrame(rows)

    # Build context string
    context = "\n\n".join(
        [f"[{r.rank}] {r.chunk_text}" for r in retrieval_df.itertuples()]
    )

    return retrieval_df, context


# Test
test_retrieval_df, test_context = retrieve(
    "What does the policy say about leave?",
    top_k=3
)

display(test_retrieval_df)
print(test_context[:1500])

,query,rank,score,source,chunk_id,chunk_text
0,What does the policy say about leave?,1,0.671852,rag-docs/HR-POL-007_General_Work_Policy.docx,rag-docs/HR-POL-007_General_Work_Policy.docx::...,restricted to authorized personnel with valid...
1,What does the policy say about leave?,2,0.626138,rag-docs/HR-POL-008_Leave_Policy.docx,rag-docs/HR-POL-008_Leave_Policy.docx::chunk_001,ABC Solutions Pvt. Ltd.\n\nABC Solutions Pvt. ...
2,What does the policy say about leave?,3,0.623699,rag-docs/HR-POL-008_Leave_Policy.docx,rag-docs/HR-POL-008_Leave_Policy.docx::chunk_006,sive unauthorized absence shall impact complia...
3,What does the policy say about leave?,4,0.597447,rag-docs/HR-POL-008_Leave_Policy.docx,rag-docs/HR-POL-008_Leave_Policy.docx::chunk_002,"ory compliance, while providing adequate rest ..."
4,What does the policy say about leave?,5,0.589947,rag-docs/HR-POL-009_Remote_Work_Policy.docx,rag-docs/HR-POL-009_Remote_Work_Policy.docx::c...,"roductivity, and compliance linkage.\n9.3\n\nH..."


[1]  restricted to authorized personnel with valid identity cards.
6.3 Violation of safety or security protocols shall be treated as serious misconduct.

7. Leave, Absence, and Punctuality

7.1 Leave shall be availed only after approval in accordance with Leave Policy (HR-POL-008).
7.2 Repeated late coming or early departure beyond three instances in a month may attract disciplinary counselling and notation in performance records under HR-POL-006.

8. Roles and Responsibilities

Role

Key Responsibilities

9. Cross-Policy Dependencies

9.1

Leave Policy (HR-POL-008):

Governs all forms of authorized and unauthorized absence.
9.2

Performance Appraisal Policy (HR-POL-006):

Attendance and conduct

[2] ABC Solutions Pvt. Ltd.

ABC Solutions Pvt. Ltd.
Document ID:

HR-POL-008

Version:

1.0

Effective Date:

January 1, 2026

Title:

Leave Policy

Classification:

Confidential – Internal Use Only

1. Objective

This policy defines the types, eligibility, accrual, utilization, and administr

## Step 8 —Reranking with an LLM


In [40]:
import re
import pandas as pd

# This function uses an LLM to rerank retrieved chunks
def rerank_results_llm(query: str, retrieval_df: pd.DataFrame, final_k: int = FINAL_K):
    
    # Step 1: Combine all retrieved chunks into a single prompt
    docs = ""
    for i, row in enumerate(retrieval_df.itertuples(index=False)):
        docs += f"\nDocument {i}:\n{row.chunk_text}\n"

    # Step 2: Create prompt for LLM
    prompt = f"""
You are a retrieval reranking assistant.

Given a query and multiple documents, return ONLY a Python list
of document numbers in order of relevance.

Query:
{query}

Documents:
{docs}

Example output:
[2, 0, 1]
"""

    # Step 3: Call LLM
    response = bedrock_runtime.converse(
        modelId=BEDROCK_LLM_MODEL_ID,
        messages=[{"role": "user", "content": [{"text": prompt}]}],
        inferenceConfig={"maxTokens": 100, "temperature": 0}
    )

    # Step 4: Extract text response
    output = response["output"]["message"]["content"][0]["text"]

    # Step 5: Extract indices from response (e.g., [2,0,1])
    indices = list(map(int, re.findall(r"\d+", output)))

    # Step 6: Reorder rows based on LLM output
    reranked_rows = []
    for idx in indices[:final_k]:
        row = retrieval_df.iloc[idx]
        reranked_rows.append(row.to_dict())

    # Convert to DataFrame
    reranked_df = pd.DataFrame(reranked_rows)

    # Add rank position
    reranked_df["rerank_position"] = range(1, len(reranked_df) + 1)

    return reranked_df


# Step 7: Test retrieval + reranking
candidate_df, _ = retrieve(
    "What is eligibility for remote work?",
    top_k=TOP_K
)

reranked_df_preview = rerank_results_llm(
    "What is eligibility for remote work?",
    candidate_df,
    final_k=FINAL_K
)

# Display reranked results
display(reranked_df_preview)


,query,rank,score,source,chunk_id,chunk_text,rerank_position
0,What is eligibility for remote work?,2,0.712409,rag-docs/HR-POL-009_Remote_Work_Policy.docx,rag-docs/HR-POL-009_Remote_Work_Policy.docx::c...,ith applicable labour laws and the Information...,1
1,What is eligibility for remote work?,1,0.728857,rag-docs/HR-POL-009_Remote_Work_Policy.docx,rag-docs/HR-POL-009_Remote_Work_Policy.docx::c...,rk:\n\nA combination of on-site and remote wor...,2
2,What is eligibility for remote work?,5,0.580958,rag-docs/HR-POL-009_Remote_Work_Policy.docx,rag-docs/HR-POL-009_Remote_Work_Policy.docx::c...,ABC Solutions Pvt. Ltd.\n\nABC Solutions Pvt. ...,3


## Step 9 — Build grounded answers with Bedrock

This is the final RAG step:
1. retrieve
2. Rerank
3. build context
4. generate a grounded answer

**Discuss with learners:**
- the answer quality depends on the context quality
- RAG does not remove hallucination risk, but it reduces it when context is good


In [41]:
import pandas as pd

# Call Bedrock LLM with a prompt and return the response text
def call_bedrock_llm(prompt: str):
    response = bedrock_runtime.converse(
        modelId=BEDROCK_LLM_MODEL_ID,
        messages=[{"role": "user", "content": [{"text": prompt}]}],
        inferenceConfig={"maxTokens": 300, "temperature": 0.2}
    )
    return response["output"]["message"]["content"][0]["text"]


# Build context string from retrieved chunks
def build_context_from_df(df: pd.DataFrame):
    return "\n\n".join(
        [f"[Source: {row.source}]\n{row.chunk_text}" for row in df.itertuples()]
    )


# Main RAG function
def answer_with_rag(query: str, retrieve_k: int = TOP_K, final_k: int = FINAL_K):
    
    # Step 1: Retrieve top chunks using embedding similarity
    retrieval_df, _ = retrieve(query, top_k=retrieve_k)
    
    # Step 2: Rerank using LLM (optional step simplified as always used)
    final_context_df = rerank_results_llm(query, retrieval_df, final_k=final_k)
    
    # Step 3: Build context from selected chunks
    context = build_context_from_df(final_context_df)
    
    # Step 4: Create final prompt for answer generation
    prompt = f"""
You are an enterprise RAG assistant.

Answer ONLY from the provided context.
If the answer is not present, say: "I don't know based on the provided context."

Context:
{context}

Question:
{query}

Answer:
"""
    
    # Step 5: Call LLM to generate final answer
    answer = call_bedrock_llm(prompt)
    
    # Step 6: Store basic run details
    run_metrics_df = pd.DataFrame([{
        "query": query,
        "retrieve_k": retrieve_k,
        "final_k": final_k,
        "n_documents": len(docs_df),
        "n_total_chunks": len(chunks_df),
        "timestamp_utc": pd.Timestamp.utcnow(),
    }])
    
    return {
        "retrieval_df": retrieval_df,          # Initial retrieved chunks
        "final_context_df": final_context_df,  # After reranking
        "answer": answer,                      # Final LLM answer
        "run_metrics_df": run_metrics_df,      # Simple run metrics
    }


# Run a demo query
demo_run = answer_with_rag("What does the policy say about leave?")

# Print final answer
print("ANSWER\n")
print(demo_run["answer"])

# Show retrieved chunks
print("\nInitial retrieval")
display(demo_run["retrieval_df"])

# Show reranked chunks
print("\nFinal context after reranking")
display(demo_run["final_context_df"])

# Show run metrics
print("\nRun metrics")
display(demo_run["run_metrics_df"])

ANSWER

The Leave Policy (HR-POL-008) defines the types, eligibility, accrual, utilization, and administration of leave for employees of ABC Solutions Pvt. Ltd. It applies to all confirmed employees across all grades and locations, governs all forms of authorized and unauthorized absence, and is administered by the Human Resources Department under the authority of the Chief Human Resources Officer (CHRO). The policy includes definitions for various types of leave such as Earned Leave (EL), Casual Leave (CL), Sick Leave (SL), and Maternity/Paternity Leave.

Initial retrieval


,query,rank,score,source,chunk_id,chunk_text
0,What does the policy say about leave?,1,0.671852,rag-docs/HR-POL-007_General_Work_Policy.docx,rag-docs/HR-POL-007_General_Work_Policy.docx::...,restricted to authorized personnel with valid...
1,What does the policy say about leave?,2,0.626138,rag-docs/HR-POL-008_Leave_Policy.docx,rag-docs/HR-POL-008_Leave_Policy.docx::chunk_001,ABC Solutions Pvt. Ltd.\n\nABC Solutions Pvt. ...
2,What does the policy say about leave?,3,0.623699,rag-docs/HR-POL-008_Leave_Policy.docx,rag-docs/HR-POL-008_Leave_Policy.docx::chunk_006,sive unauthorized absence shall impact complia...
3,What does the policy say about leave?,4,0.597447,rag-docs/HR-POL-008_Leave_Policy.docx,rag-docs/HR-POL-008_Leave_Policy.docx::chunk_002,"ory compliance, while providing adequate rest ..."
4,What does the policy say about leave?,5,0.589947,rag-docs/HR-POL-009_Remote_Work_Policy.docx,rag-docs/HR-POL-009_Remote_Work_Policy.docx::c...,"roductivity, and compliance linkage.\n9.3\n\nH..."



Final context after reranking


,query,rank,score,source,chunk_id,chunk_text,rerank_position
0,What does the policy say about leave?,2,0.626138,rag-docs/HR-POL-008_Leave_Policy.docx,rag-docs/HR-POL-008_Leave_Policy.docx::chunk_001,ABC Solutions Pvt. Ltd.\n\nABC Solutions Pvt. ...,1
1,What does the policy say about leave?,1,0.671852,rag-docs/HR-POL-007_General_Work_Policy.docx,rag-docs/HR-POL-007_General_Work_Policy.docx::...,restricted to authorized personnel with valid...,2
2,What does the policy say about leave?,4,0.597447,rag-docs/HR-POL-008_Leave_Policy.docx,rag-docs/HR-POL-008_Leave_Policy.docx::chunk_002,"ory compliance, while providing adequate rest ...",3



Run metrics


,query,retrieve_k,final_k,n_documents,n_total_chunks,timestamp_utc
0,What does the policy say about leave?,5,3,9,81,2026-04-23 18:10:59.605882+00:00


# RAG pipeline step-wise breakdown

| Stage | What happens | Model Type | # Calls | Token Usage | Cost Driver | When it runs |
|------|-------------|------------|--------|-------------|-------------|--------------|
| Data Ingestion |  |  |  |  |  |  |
| Text Extraction |  |  |  |  |  |  |
| Chunking |  |  |  |  |  |  |
| Embedding Generation |  |  |  |  |  |  |
| Storage (Vector DB) |  |  |  |  |  |  |
| Query Embedding |  |  |  |  |  |  |
| Retrieval (Similarity Search) |  |  |  |  |  |  |
| Reranking (optional) |  |  |  |  |  |  |
| Context Building |  |  |  |  |  |  |
| Final Answer Generation |  |  |  |  |  |  |

# RAG Evaluation

In this section, we evaluate the RAG pipeline using metrics that work reliably in this notebook environment.

We will calculate:

- **Exact Match**: checks whether the predicted answer exactly matches the reference answer
- **ROUGE**: checks text overlap between predicted answer and reference answer


These metrics help us evaluate both:
- **answer quality**
- **retrieval quality**



## ROUGE Score (Evaluation of Generated Text)

ROUGE (Recall-Oriented Understudy for Gisting Evaluation) is a metric used to compare a **generated answer** with a **reference (ground truth) answer**.

It measures how much overlap exists between the two texts.

### Types of ROUGE used

- **ROUGE-1**  
  Measures overlap of individual words (unigrams)  
  → “How many words match?”

- **ROUGE-2**  
  Measures overlap of word pairs (bigrams)  
  → “How many word sequences match?”

- **ROUGE-L**  
  Measures longest common sequence of words  
  → “How similar is the overall structure?”

### What does the score mean?

ROUGE values range from **0 to 1**:
- **1.0** → perfect match
- **0.0** → no overlap

Higher ROUGE score means the generated answer is closer to the reference answer.

### Important Note

ROUGE checks **text similarity**, not correctness:
- A high score means answers look similar
- It does NOT guarantee the answer is factually correct

---

## Why Accuracy, Precision, and F1 Score are NOT used here

Accuracy, Precision, and F1 Score are designed for **classification tasks**, where outputs are from a fixed set of labels.

Examples:
- Yes / No  
- Category A / B / C  
- Relevant / Not Relevant  

In this notebook, the model generates **free-text answers**, not labels.

Example:
- Ground truth: "Employees are eligible after 6 months"
- Model output: "Eligibility starts after six months of employment"

These are **correct but not exact matches**, so:
- Accuracy → would mark this as wrong ❌  
- F1 / Precision → not meaningful for sentence-level text  

---

## Why ROUGE is suitable for RAG

ROUGE works better because:
- It measures **partial overlap**
- It captures **similar meaning expressed differently**
- It is designed for **text generation tasks**

---

## Summary

- Use **Accuracy / F1 / Precision** → for classification problems  
- Use **ROUGE** → for text generation problems (like RAG)  

In [82]:
import io
import pandas as pd
import boto3
from rouge_score import rouge_scorer

# Set S3 details
S3_BUCKET = "s3-demo-bucket-adp"
S3_KEY = "rag_evaluation_qna_15.csv"

# Create S3 client and read file
s3 = boto3.client("s3")
response = s3.get_object(Bucket=S3_BUCKET, Key=S3_KEY)
eval_df = pd.read_csv(io.BytesIO(response["Body"].read()))

# Show the loaded file
display(eval_df.head())
print("Shape:", eval_df.shape)
print("Columns:", eval_df.columns.tolist())

,question,answer,grounding_reference
0,Within how many working days of the Date of Jo...,Employees must be enrolled under EPF and ESI w...,HR-POL-001 | Clause 4.2.1
1,What happens if background verification advers...,Unresolved adverse background verification fin...,HR-POL-001 | Clause 4.3.2
2,What is the minimum assessment score required ...,A minimum assessment score of 70 percent is re...,HR-POL-001 | Clause 4.5.2
3,Which protected characteristics are covered un...,"Protected characteristics include gender, mari...",HR-POL-002 | Clause 3.1


Shape: (4, 3)
Columns: ['question', 'answer', 'grounding_reference']


### Run RAG for Each Question

In [83]:
# Store model answers and retrieved contexts
predicted_answers = []


# Run the RAG pipeline for each question
for question in eval_df["question"]:
    result = answer_with_rag(question)
    predicted_answers.append(result["answer"])


# Add model outputs to the evaluation DataFrame
eval_df["predicted_answer"] = predicted_answers


# Show sample output
display(eval_df.head())

,question,answer,grounding_reference,predicted_answer
0,Within how many working days of the Date of Jo...,Employees must be enrolled under EPF and ESI w...,HR-POL-001 | Clause 4.2.1,Within 7 working days of the Date of Joining (...
1,What happens if background verification advers...,Unresolved adverse background verification fin...,HR-POL-001 | Clause 4.3.2,Adverse findings not resolved within 30 days m...
2,What is the minimum assessment score required ...,A minimum assessment score of 70 percent is re...,HR-POL-001 | Clause 4.5.2,A minimum assessment score of 70% is required ...
3,Which protected characteristics are covered un...,"Protected characteristics include gender, mari...",HR-POL-002 | Clause 3.1,The protected characteristics covered under th...


### Exact Match

In [84]:
# Normalize text before comparison
def normalize_text(text):
    return str(text).strip().lower()

# Exact Match = 1 if answer and predicted answer are exactly the same after normalization
eval_df["exact_match"] = (
    eval_df["answer"].apply(normalize_text) ==
    eval_df["predicted_answer"].apply(normalize_text)
).astype(int)

print("Exact Match Score:", round(eval_df["exact_match"].mean(), 4))

Exact Match Score: 0.0


### ROUGE Scores

In [85]:
# Create ROUGE scorer
scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)

# Store ROUGE scores for each row
rouge_rows = []

for row in eval_df.itertuples(index=False):
    scores = scorer.score(str(row.answer), str(row.predicted_answer))
    
    rouge_rows.append({
        "question": row.question,
        "rouge1_fmeasure": scores["rouge1"].fmeasure,
        "rouge2_fmeasure": scores["rouge2"].fmeasure,
        "rougeL_fmeasure": scores["rougeL"].fmeasure,
    })

# Convert to DataFrame
rouge_df = pd.DataFrame(rouge_rows)

# Show sample scores
display(rouge_df.head())

# Print average ROUGE scores
print("Average ROUGE-1:", round(rouge_df["rouge1_fmeasure"].mean(), 4))
print("Average ROUGE-2:", round(rouge_df["rouge2_fmeasure"].mean(), 4))
print("Average ROUGE-L:", round(rouge_df["rougeL_fmeasure"].mean(), 4))

,question,rouge1_fmeasure,rouge2_fmeasure,rougeL_fmeasure
0,Within how many working days of the Date of Jo...,0.629630,0.576923,0.333333
1,What happens if background verification advers...,0.727273,0.571429,0.727273
2,What is the minimum assessment score required ...,0.888889,0.720000,0.888889
3,Which protected characteristics are covered un...,0.693878,0.510638,0.693878


Average ROUGE-1: 0.7349
Average ROUGE-2: 0.5947
Average ROUGE-L: 0.6608


### Context Hit Rate and Final Summary

In [86]:

# Final summary table
summary_df = pd.DataFrame([{
    "exact_match": round(eval_df["exact_match"].mean(), 4),
    "avg_rouge1": round(rouge_df["rouge1_fmeasure"].mean(), 4),
    "avg_rouge2": round(rouge_df["rouge2_fmeasure"].mean(), 4),
    "avg_rougeL": round(rouge_df["rougeL_fmeasure"].mean(), 4),
}])

display(summary_df)

,exact_match,avg_rouge1,avg_rouge2,avg_rougeL
0,0.0,0.7349,0.5947,0.6608
